# Phase 15 — Robustness Testing

This notebook validates the authored challenge set, reviews the latest saved robustness run, and keeps the automatic perturbation, information-loss, and semantic challenge tracks separate. It does not load the official test split or train models. The full evaluation cell is opt-in.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from deeptext.experiments.robustness import (
    TRANSFORMATION_TYPES,
    generate_automatic_perturbations,
    validate_semantic_challenge,
)

ROOT = Path.cwd()
CHALLENGE_PATH = ROOT / "data/robustness/semantic_challenge.csv"
OUTPUT_ROOT = ROOT / "outputs/robustness"

## Challenge-set integrity

The semantic challenge set is author-created, balanced, and not independently adjudicated. Its scores are descriptive and must not be interpreted as final test results.

In [ ]:
challenge = pd.read_csv(CHALLENGE_PATH)
challenge_summary = validate_semantic_challenge(challenge)
assert challenge_summary["sample_count"] == 200
assert challenge_summary["class_distribution"] == {"0": 100, "1": 100}
display(
    pd.DataFrame(
        challenge_summary["category_distribution"].items(),
        columns=["category", "examples"],
    )
)
challenge_summary

## Inspect automatic perturbations

This deterministic preview uses a tiny synthetic pair and does not access IMDb data or model artifacts. Label-preserving status is a conservative heuristic, not a guarantee.

In [ ]:
preview_source = pd.DataFrame(
    [
        {
            "review_id": "preview/pos/1",
            "label": 1,
            "text": "A wonderful movie with excellent performances.",
        },
        {
            "review_id": "preview/neg/1",
            "label": 0,
            "text": "A terrible movie with boring performances.",
        },
    ]
)
preview, preview_rejections = generate_automatic_perturbations(
    preview_source, seed=42, max_variants_per_review=len(TRANSFORMATION_TYPES)
)
display(
    preview[
        [
            "review_id",
            "category",
            "transformation_type",
            "original_text",
            "transformed_text",
        ]
    ].head(12)
)
print(
    f"{len(preview)} accepted variants; "
    f"{len(preview_rejections)} candidates rejected by QC."
)

## Load the latest completed run

The analysis compares each automatic variant with its matching original validation review. Category and overall summaries aggregate at review level; pooled variant metrics are labeled descriptive. Truncation and sentence ablation have their own information-loss reports.

In [ ]:
completed_runs = sorted(
    path
    for path in OUTPUT_ROOT.glob("imdb-robustness-*")
    if (path / "metadata.json").is_file() and (path / "metrics.json").is_file()
)
if not completed_runs:
    raise FileNotFoundError(
        "No completed Phase 15 run found. Run the evaluation command first."
    )
RUN_DIR = completed_runs[-1]
metadata = json.loads((RUN_DIR / "metadata.json").read_text(encoding="utf-8"))
metrics = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
assert metadata["official_test_dataset_loaded"] is False
assert metadata["phase13_test_predictions_used"] is False
assert metadata["model_training_or_tuning_performed"] is False
assert metadata["source_artifacts_unchanged"] is True
print(RUN_DIR)
print("Validation source reviews:", metadata["validation_split"]["sampled_rows"])
print("Accepted automatic variants:", metadata["automatic_track"]["accepted_variants"])
print("Semantic challenge examples:", metadata["semantic_challenge"]["sample_count"])

In [ ]:
automatic_summary = pd.read_csv(
    RUN_DIR / "automatic_perturbations/transformation_summary.csv"
)
display(
    automatic_summary[automatic_summary.summary_level == "overall"][
        [
            "model",
            "unique_review_count",
            "original_accuracy_matched_reviews",
            "robustness_accuracy_review_level",
            "accuracy_degradation",
            "robustness_f1_review_level",
            "prediction_flip_rate",
        ]
    ]
)
display(
    automatic_summary[automatic_summary.summary_level == "category"][
        [
            "model",
            "group",
            "unique_review_count",
            "robustness_accuracy_review_level",
            "accuracy_degradation",
            "prediction_flip_rate",
        ]
    ].head(20)
)
length_summary = pd.read_csv(RUN_DIR / "automatic_perturbations/length_summary.csv")
display(
    length_summary[
        [
            "model",
            "group",
            "unique_review_count",
            "robustness_accuracy_review_level",
            "accuracy_degradation",
        ]
    ].head(20)
)

## Semantic challenge results and failures

In [ ]:
semantic_summary = pd.read_csv(RUN_DIR / "semantic_challenge/category_summary.csv")
display(semantic_summary.pivot(index="category", columns="model", values="accuracy"))
semantic_failures = pd.read_csv(RUN_DIR / "semantic_challenge/failure_examples.csv")
display(
    semantic_failures[
        [
            "sample_id",
            "category",
            "model",
            "true_label",
            "predicted_label",
            "text",
            "rationale",
        ]
    ].head(20)
)
pairwise = pd.read_csv(RUN_DIR / "cross_model/pairwise_agreement.csv")
display(pairwise[pairwise.track == "semantic_challenge"])

## Saved plots and source-artifact integrity

In [ ]:
for plot_name in metadata["plot_files"]:
    display(Image(filename=str(RUN_DIR / "plots" / plot_name)))
print(
    "All frozen model files match their before/after SHA256 records:",
    metadata["source_artifacts_unchanged"],
)

## Optional full rerun

Leave `RUN_FULL_EVALUATION` disabled to review saved outputs only. Enabling it performs the complete validation-only analysis and writes a new timestamped output directory.

In [ ]:
RUN_FULL_EVALUATION = False
if RUN_FULL_EVALUATION:
    from deeptext.experiments.robustness import run_robustness_evaluation

    rerun = run_robustness_evaluation(device="auto")
    print(rerun["output_directory"])